In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

In [5]:
# ============================================================
# STEP 1: LOAD DATA
# ============================================================
import pandas as pd

file_path = "dataset/NIFTY_50_COMPANIES.csv"   # change if needed
df = pd.read_csv(file_path)

print("="*60)
print("BASIC INFORMATION")
print("="*60)

# Shape
print(f"Shape (Rows, Columns): {df.shape}")

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

# ============================================================
# STEP 2: DATE INFORMATION
# ============================================================
print("\n" + "="*60)
print("DATE INFORMATION")
print("="*60)

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

print(f"Start Date : {df['Date'].min()}")
print(f"End Date   : {df['Date'].max()}")
print(f"Total Unique Dates : {df['Date'].nunique()}")

# ============================================================
# STEP 3: TICKER INFORMATION
# ============================================================
print("\n" + "="*60)
print("TICKER INFORMATION")
print("="*60)

if 'Ticker' in df.columns:
    print(f"Total Unique Tickers : {df['Ticker'].nunique()}")
    print("Ticker List:")
    print(df['Ticker'].unique())

# ============================================================
# STEP 4: MISSING VALUES
# ============================================================
print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)

missing = df.isnull().sum()
print(missing[missing > 0])

# ============================================================
# STEP 5: NUMERICAL SUMMARY
# ============================================================
print("\n" + "="*60)
print("NUMERICAL SUMMARY")
print("="*60)

print(df.describe())

# ============================================================
# STEP 6: SAMPLE DATA
# ============================================================
print("\n" + "="*60)
print("FIRST 5 ROWS")
print("="*60)
print(df.head())

print("\n" + "="*60)
print("LAST 5 ROWS")
print("="*60)
print(df.tail())

BASIC INFORMATION
Shape (Rows, Columns): (304543, 19)

Columns:
['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'MACD', 'Signal_Line', 'RSI_14', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'Daily_Return_%']

Data Types:
Date                  str
Adj Close         float64
Close             float64
High              float64
Low               float64
Open              float64
Volume              int64
Ticker                str
SMA_20            float64
SMA_50            float64
EMA_12            float64
EMA_26            float64
MACD              float64
Signal_Line       float64
RSI_14            float64
BB_Mid            float64
BB_Upper          float64
BB_Lower          float64
Daily_Return_%    float64
dtype: object

DATE INFORMATION
Start Date : 1997-07-01 00:00:00
End Date   : 2025-11-11 00:00:00
Total Unique Dates : 7109

TICKER INFORMATION
Total Unique Tickers : 51
Ticker List:
<StringArray>
[  'RELIANCE.NS',        'TCS.NS'

In [6]:
# ============================================================
# STEP 2: DATA CLEANING
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: DATA CLEANING")
print("=" * 60)

before = len(df)
print(f"Shape before: {df.shape}")
null_counts = df.isnull().sum()
print(f"Null counts:\n{null_counts[null_counts > 0].to_string()}")
df = df.dropna()
print(f"Rows dropped (NaN from indicator lookback): {before - len(df)}")
print(f"Shape after : {df.shape}")


STEP 2: DATA CLEANING
Shape before: (304543, 19)
Null counts:
SMA_20             969
SMA_50            2499
RSI_14            2697
BB_Mid             969
BB_Upper           969
BB_Lower           969
Daily_Return_%      51
Rows dropped (NaN from indicator lookback): 4533
Shape after : (300010, 19)


In [7]:
# ============================================================
# STEP 3: FEATURE ENGINEERING
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: FEATURE ENGINEERING")
print("=" * 60)

def engineer_features(grp):
    g = grp.copy()
    g['Return']        = g['Adj Close'].pct_change()
    g['Return_Lag1']   = g['Return'].shift(1)
    g['Return_Lag2']   = g['Return'].shift(2)
    g['Return_Lag3']   = g['Return'].shift(3)
    g['Volatility_10'] = g['Return'].rolling(10).std()
    g['SMA_ratio']     = g['Close'] / g['SMA_20']
    g['EMA_ratio']     = g['Close'] / g['EMA_12']
    g['MACD_diff']     = g['MACD'] - g['Signal_Line']
    g['RSI_norm']      = g['RSI_14'] / 100
    denom              = (g['BB_Upper'] - g['BB_Lower']).replace(0, np.nan)
    g['BB_position']   = (g['Close'] - g['BB_Lower']) / denom
    return g

print("Engineering features per ticker...")
chunks = [engineer_features(df[df['Ticker'] == t]) for t in df['Ticker'].unique()]
df = pd.concat(chunks, ignore_index=True).sort_values(['Ticker','Date']).reset_index(drop=True)

eng_features = ['Return','Return_Lag1','Return_Lag2','Return_Lag3',
                'Volatility_10','SMA_ratio','EMA_ratio','MACD_diff','RSI_norm','BB_position']
print(f"Features added: {eng_features}")
print(f"\nFeature Stats:")
print(df[eng_features].describe().round(4).to_string())


STEP 3: FEATURE ENGINEERING
Engineering features per ticker...
Features added: ['Return', 'Return_Lag1', 'Return_Lag2', 'Return_Lag3', 'Volatility_10', 'SMA_ratio', 'EMA_ratio', 'MACD_diff', 'RSI_norm', 'BB_position']

Feature Stats:
            Return  Return_Lag1  Return_Lag2  Return_Lag3  Volatility_10    SMA_ratio    EMA_ratio    MACD_diff     RSI_norm  BB_position
count  299959.0000  299908.0000  299857.0000  299806.0000    299500.0000  300010.0000  300010.0000  300010.0000  300010.0000  300010.0000
mean        0.0020       0.0020       0.0020       0.0020         0.0237       1.0074       1.0035       0.0001       0.5251       0.5446
std         0.2140       0.2141       0.2141       0.2141         0.2133       0.0783       0.0474      15.1818       0.1708       0.3247
min        -0.9909      -0.9909      -0.9909      -0.9909         0.0000       0.0749       0.0662    -525.9546       0.0000      -0.5621
25%        -0.0102      -0.0102      -0.0102      -0.0102         0.0125   